<a href="https://colab.research.google.com/github/beingAnujChaudhary/DSFS-Joel-Grus/blob/main/notebooks/chapter_01_introduction/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/DSFS-Joel-Grus.git

# Move into repository
%cd /content/DSFS-Joel-Grus

# Move into Chapter 1 folder
%cd notebooks/chapter_01_introduction

Mounted at /content/drive
Cloning into 'DSFS-Joel-Grus'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 84 (delta 48), reused 57 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 828.74 KiB | 4.48 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/DSFS-Joel-Grus
/content/DSFS-Joel-Grus/notebooks/chapter_01_introduction


# Chapter 1: Introduction — Exercises

**Goal**: Reinforce core concepts from Chapter 1 by extending the original examples, experimenting with parameters, and building small analytical tools from scratch.

**Rule**: Try solving each exercise yourself before looking at the solution. Write the code manually.

In [3]:
# Cell 1: Setup & Data Reuse
from collections import Counter, defaultdict
import math

# Re-declare chapter data for self-containment
users = [
    {"id": 0, "name": "Hero"}, {"id": 1, "name": "Dunn"},
    {"id": 2, "name": "Sue"},  {"id": 3, "name": "Chi"},
    {"id": 4, "name": "Thor"}, {"id": 5, "name": "Clive"},
    {"id": 6, "name": "Hicks"}, {"id": 7, "name": "Devin"},
    {"id": 8, "name": "Kate"}, {"id": 9, "name": "Klein"}
]

friendships = [
    (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
    (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)
]

for user in users:
    user["friends"] = []
for i, j in friendships:
    users[i]["friends"].append(users[j])
    users[j]["friends"].append(users[i])

interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"), (3, "R"), (3, "Python"),
    (3, "statistics"), (3, "regression"), (3, "probability"),
    (4, "machine learning"), (4, "regression"), (4, "decision trees"),
    (4, "libsvm"), (5, "Python"), (5, "R"), (5, "Java"), (5, "C++"),
    (5, "Haskell"), (5, "programming languages"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"), (9, "Hadoop"),
    (9, "Java"), (9, "MapReduce"), (9, "Big Data")
]

salaries_and_tenures = [
    (83000, 8.7), (88000, 8.1), (48000, 0.7), (76000, 6),
    (69000, 6.5), (76000, 7.5), (60000, 2.5), (83000, 10),
    (48000, 1.9), (63000, 4.2)
]

experience_paid_actual = [
    (0.7, "paid"), (1.9, "unpaid"), (2.5, "paid"), (4.2, "unpaid"),
    (6.0, "unpaid"), (6.5, "unpaid"), (7.5, "unpaid"), (8.1, "unpaid"),
    (8.7, "paid"), (10.0, "paid")
]

print("✅ Data loaded successfully")

✅ Data loaded successfully


## Exercise 1: Weighted Connection Suggestions

**Problem**: The original `friends_of_friend_ids` function counts mutual friends equally. Modify it to return a `Counter` where each potential connection is weighted by **how many mutual friends they share**. Filter out self and existing friends.

**Hint**: Use a nested loop over friends, then over their friends, and increment a counter.

<details>
<summary>Click to view solution</summary>

### Solution

```python
# Cell 2: Exercise 1 Solution
def not_the_same(user, other_user):
    return user["id"] != other_user["id"]

def not_friends(user, other_user):
    return all(not_the_same(friend, other_user) for friend in user["friends"])

def weighted_friend_suggestions(user):
    """Returns Counter of {potential_friend_id: mutual_friend_count}"""
    suggestions = Counter()
    for friend in user["friends"]:
        for foaf in friend["friends"]:
            if not_the_same(user, foaf) and not_friends(user, foaf):
                suggestions[foaf["id"]] += 1
    return suggestions

# Test on Chi (id 3)
chi = users[3]
print("Weighted suggestions for Chi:", weighted_friend_suggestions(chi))
# Expected: Counter({0: 2, 5: 1})
```

</details>

## Exercise 2: Combined Friend + Interest Scoring

**Problem**: Create a function `suggest_connections_combined(user)` that scores potential connections using **both** mutual friends and mutual interests.
- Mutual friends count as 2 points each
- Shared interests count as 1 point each
- Return a list of `(user_id, score)` sorted descending. Exclude self and existing friends.

**Hint**: Build two `Counter`s and add them together.

<details>
<summary>Click to view solution</summary>
### Solution

```python

# Build interest indexes
user_ids_by_interest = defaultdict(list)
interests_by_user_id = defaultdict(list)
for uid, interest in interests:
    user_ids_by_interest[interest].append(uid)
    interests_by_user_id[uid].append(interest)

def shared_interests_count(user):
    return Counter(
        other_uid
        for interest in interests_by_user_id[user["id"]]
        for other_uid in user_ids_by_interest[interest]
        if other_uid != user["id"]
    )

def suggest_connections_combined(user):
    friend_score = weighted_friend_suggestions(user)
    interest_score = shared_interests_count(user)
    
    # Combine scores
    combined = friend_score.copy()
    for uid, score in interest_score.items():
        combined[uid] += score
        
    # Filter existing friends & self
    friend_ids = {f["id"] for f in user["friends"]} | {user["id"]}
    return sorted(
        [(uid, score) for uid, score in combined.items() if uid not in friend_ids],
        key=lambda x: x[1],
        reverse=True
    )

print("Combined suggestions for Hero (id 0):")
for uid, score in suggest_connections_combined(users[0]):
    print(f"  User {uid}: {score} points")
    
```
</details>

## Exercise 3: Dynamic Salary Bucketing

**Problem**: The book hardcodes tenure buckets at `<2`, `2-5`, and `>5`. Write a function `bucketed_salary_stats(data, thresholds)` that:
- Accepts a list of `(salary, tenure)` pairs
- Accepts a sorted list of thresholds (e.g., `[1, 3, 7]`)
- Returns a dict mapping bucket labels to `(count, avg_salary)`
- Buckets should be: `"less than X"`, `"X to Y"`, `"Y+"`

**Test with thresholds `[1, 3, 7]`**

<details>
<summary>Click to view solution</summary>


```python
# Cell 4: Exercise 3 Solution
def bucketed_salary_stats(data, thresholds):
    # Create bucket labels
    labels = []
    for i, t in enumerate(thresholds):
        if i == 0:
            labels.append(f"less than {t}")
        else:
            labels.append(f"{thresholds[i-1]} to {t}")
    labels.append(f"{thresholds[-1]}+")
    
    buckets = {label: [] for label in labels}
    
    for salary, tenure in data:
        # Find bucket index
        idx = 0
        for i, t in enumerate(thresholds):
            if tenure < t:
                idx = i
                break
        else:
            idx = len(thresholds)  # Last bucket
            
        buckets[labels[idx]].append(salary)
        
    # Compute stats
    return {
        label: (len(sals), sum(sals)/len(sals) if sals else 0)
        for label, sals in buckets.items()
    }

stats = bucketed_salary_stats(salaries_and_tenures, [1, 3, 7])
for bucket, (count, avg) in stats.items():
    print(f"{bucket}: {count} users, avg ${avg:,.0f}")

```
</details>

## Exercise 4: Optimizing Prediction Thresholds

**Problem**: The rule-based classifier uses hardcoded thresholds `3.0` and `8.5`. Write a function `find_optimal_thresholds(data)` that:
- Tests all possible threshold pairs `(t1, t2)` where `0 < t1 < t2 < 12` in steps of `0.5`
- Predicts `"paid"` if `exp < t1` or `exp >= t2`, else `"unpaid"`
- Returns the `(t1, t2)` pair that minimizes misclassification error

**Hint**: Use nested loops and count mismatches.

<details>
<summary>Click to view solution</summary>

```python
# Cell 5: Exercise 4 Solution
def predict_with_thresholds(exp, t1, t2):
    if exp < t1 or exp >= t2:
        return "paid"
    return "unpaid"

def find_optimal_thresholds(data):
    best_error = float('inf')
    best_t1, best_t2 = 0, 0
    
    t1_vals = [round(x*0.5, 1) for x in range(1, 24)]  # 0.5 to 11.5
    t2_vals = [round(x*0.5, 1) for x in range(2, 25)]  # 1.0 to 12.0
    
    for t1 in t1_vals:
        for t2 in t2_vals:
            if t1 >= t2:
                continue
            errors = sum(1 for exp, actual in data if predict_with_thresholds(exp, t1, t2) != actual)
            if errors < best_error:
                best_error = errors
                best_t1, best_t2 = t1, t2
                
    return best_t1, best_t2, best_error

t1, t2, err = find_optimal_thresholds(experience_paid_actual)
print(f"Optimal thresholds: t1={t1}, t2={t2} | Misclassifications: {err}/{len(experience_paid_actual)}")

```
</details>

## Exercise 5: Stop-Word Filtering for Topic Analysis

**Problem**: The original word frequency counts all words, including common ones. Implement a simple stop-word filter:
- Ignore words shorter than 4 characters
- Ignore a predefined stop-word list: `{"data", "big", "and", "for", "the"}`
- Return the top 5 filtered words with counts

**Why?**: Real-world text analysis always requires noise reduction.

<details>
<summary>Click to view solution</summary>

```python
# Cell 6: Exercise 5 Solution
STOP_WORDS = {"data", "big", "and", "for", "the", "is", "in", "on"}

def filtered_word_counts(interest_data, min_len=4):
    counts = Counter()
    for _, interest in interest_data:
        words = interest.lower().split()
        for word in words:
            if len(word) >= min_len and word not in STOP_WORDS:
                counts[word] += 1
    return counts

top_words = filtered_word_counts(interests).most_common(5)
print("Top 5 filtered topic words:")
for word, count in top_words:
    print(f"  {word}: {count}")
    
```
</details>

## 📝 Exercise Summary & Reflection

| Exercise | Concept Practiced | Key Takeaway |
|----------|-------------------|--------------|
| 1 | `Counter` aggregation | Simple counting reveals network structure |
| 2 | Multi-signal scoring | Combining signals improves recommendation quality |
| 3 | Parameterized bucketing | Hardcoded thresholds hide patterns; experiment dynamically |
| 4 | Grid search optimization | Rule-based models can be tuned systematically |
| 5 | Text preprocessing | Raw text is noisy; filtering is essential for real analysis |

**Next Step**: Try modifying the thresholds in Ex 3 & 4, or add your own stop-words in Ex 5. Notice how small changes impact results.

